In [2]:
import pandas as pd
df_train = pd.read_json('training_t5.jsonl', lines = True)
df_train

,input,output
0,all cars are a type of vehicle,A car vehicle
1,nothing that is a soda is a juice,E soda juice
2,everything that is a planet is a celestial body,A planet celestial_body
3,every cat is an invisible creature,A cat creature
4,there are no capital cities which are oceans,E capital_cities ocean
...,...,...
2475,it can be deduced that some water is not liqu...,O water liquid
2476,therefore it is argued that there are no dogs...,O dog animal
2477,for this reason every single sparrow is a bird .,A sparrow bird
2478,it must logically follow that some elephants ...,I elephant mammal


In [3]:
from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments
from datasets import Dataset

model_name = "google-t5/t5-small"

tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

dataset = Dataset.from_pandas(df_train)

def preprocess(batch):
    inputs = tokenizer(batch["input"], padding="max_length", truncation=True, max_length=64)
    labels = tokenizer(batch["output"], padding="max_length", truncation=True, max_length=16)
    inputs["labels"] = labels["input_ids"]
    return inputs

dataset = dataset.map(preprocess, batched=True, remove_columns=["input", "output"])

training_args = TrainingArguments(
    output_dir="./t5_logic",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    num_train_epochs=10,
    learning_rate=3e-4,
    logging_steps=50,
    save_strategy="epoch",
    report_to="none",
    fp16=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset
)

trainer.train()


You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


Map:   0%|          | 0/2480 [00:00<?, ? examples/s]

c:\Users\Logesh SD\OneDrive - Delict Technology Services Private Ltd\COURSE CONTENT\Desktop\VSCode projects\.venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss
50,2.105300
100,0.243000
150,0.184300
200,0.149500
250,0.140000
300,0.122800
350,0.113700
400,0.102700
450,0.097300
500,0.091700


c:\Users\Logesh SD\OneDrive - Delict Technology Services Private Ltd\COURSE CONTENT\Desktop\VSCode projects\.venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\Logesh SD\OneDrive - Delict Technology Services Private Ltd\COURSE CONTENT\Desktop\VSCode projects\.venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\Logesh SD\OneDrive - Delict Technology Services Private Ltd\COURSE CONTENT\Desktop\VSCode projects\.venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
c:\Users\Logesh SD\OneDrive - Delict Technology Services P

TrainOutput(global_step=780, training_loss=0.24492238209797787, metrics={'train_runtime': 3631.414, 'train_samples_per_second': 6.829, 'train_steps_per_second': 0.215, 'total_flos': 419559584563200.0, 'train_loss': 0.24492238209797787, 'epoch': 10.0})

In [4]:
df_inp = pd.read_json("train_data.json")
df_inp

,id,syllogism,validity,plausibility
0,50146f21-d265-4e3a-8d93-8165cdbe89a3,All cars are a type of vehicle. No animal is a...,False,True
1,dfafb4f6-4e1d-4cd5-aeb4-75d36aafdf1a,Nothing that is a soda is a juice. A portion o...,True,True
2,e30b1f83-a4c3-49cb-8aaf-5f64208c625b,Everything that is a planet is a celestial bod...,False,False
3,a30e07d5-0fb3-4097-9892-4b145b0c54f5,Every cat is an invisible creature. A number o...,True,False
4,5b8161b7-b1bf-4e16-a854-cd52cdce8a1b,There are no capital cities which are oceans. ...,True,True
...,...,...,...,...
955,6df0ab4c-8518-4efb-98d5-e1bd172b5a35,Some creatures that can fly are birds. It is t...,False,True
956,0db674a5-4391-43fd-8d40-b6ec2b6e77e3,Every single thing that is a hammer is a tool....,False,True
957,6650eef5-2045-4ee2-8fe1-a3036d54c6b6,No person who is a doctor is also a teacher. A...,True,True
958,452c2309-abd1-4cd9-9cd9-1e693ef85cf6,A creature that is a reptile is never a mammal...,True,False


In [5]:
df_inp["premise1"] = df_inp["syllogism"].str.split(".").str[0]
df_inp["premise2"] = df_inp["syllogism"].str.split(".").str[1]
df_inp["conclusion"] = df_inp["syllogism"].str.split(".").str[2]

In [6]:
df_inp

,id,syllogism,validity,plausibility,premise1,premise2,conclusion
0,50146f21-d265-4e3a-8d93-8165cdbe89a3,All cars are a type of vehicle. No animal is a...,False,True,All cars are a type of vehicle,No animal is a car,"Therefore, no animal can be a vehicle"
1,dfafb4f6-4e1d-4cd5-aeb4-75d36aafdf1a,Nothing that is a soda is a juice. A portion o...,True,True,Nothing that is a soda is a juice,A portion of the things that are beverages ar...,The only logical conclusion is that some beve...
2,e30b1f83-a4c3-49cb-8aaf-5f64208c625b,Everything that is a planet is a celestial bod...,False,False,Everything that is a planet is a celestial body,Anything that is a sun is a celestial body,There exists at least one sun that is a planet
3,a30e07d5-0fb3-4097-9892-4b145b0c54f5,Every cat is an invisible creature. A number o...,True,False,Every cat is an invisible creature,A number of cats are animals,"Consequently, a portion of animals are invisible"
4,5b8161b7-b1bf-4e16-a854-cd52cdce8a1b,There are no capital cities which are oceans. ...,True,True,There are no capital cities which are oceans,A select few large cities are classified as c...,"Therefore, it is clear that some large cities..."
...,...,...,...,...,...,...,...
955,6df0ab4c-8518-4efb-98d5-e1bd172b5a35,Some creatures that can fly are birds. It is t...,False,True,Some creatures that can fly are birds,It is true that some sparrows are flying crea...,"For this reason, every single sparrow is a bird"
956,0db674a5-4391-43fd-8d40-b6ec2b6e77e3,Every single thing that is a hammer is a tool....,False,True,Every single thing that is a hammer is a tool,All screwdrivers are also tools,It must be true that no screwdriver is a hammer
957,6650eef5-2045-4ee2-8fe1-a3036d54c6b6,No person who is a doctor is also a teacher. A...,True,True,No person who is a doctor is also a teacher,Anything that is a teacher is a profession,"Therefore, some professions are not doctors"
958,452c2309-abd1-4cd9-9cd9-1e693ef85cf6,A creature that is a reptile is never a mammal...,True,False,A creature that is a reptile is never a mammal,There is a group of reptiles that are also el...,It must logically follow that some elephants ...


In [7]:
from transformers import T5Tokenizer, T5ForConditionalGeneration
import torch

BASE_MODEL = "google-t5/t5-small"
CHECKPOINT = "./t5_logic/checkpoint-780"  


tokenizer = T5Tokenizer.from_pretrained(BASE_MODEL)


model = T5ForConditionalGeneration.from_pretrained(CHECKPOINT)

model.eval()


def to_canonical(sentence, max_len=16):
    input_text = sentence.strip()

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_length=max_len,
            num_beams=4,
            early_stopping=True
        )

    decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return decoded.strip()

df_inp["can_1"] = df_inp["premise1"].apply(to_canonical)
df_inp["can_2"] = df_inp["premise2"].apply(to_canonical)
df_inp["can_conc"] = df_inp["conclusion"].apply(to_canonical)


In [8]:
df_inp

,id,syllogism,validity,plausibility,premise1,premise2,conclusion,can_1,can_2,can_conc
0,50146f21-d265-4e3a-8d93-8165cdbe89a3,All cars are a type of vehicle. No animal is a...,False,True,All cars are a type of vehicle,No animal is a car,"Therefore, no animal can be a vehicle",A car vehicle,E animal animal,E animal animal
1,dfafb4f6-4e1d-4cd5-aeb4-75d36aafdf1a,Nothing that is a soda is a juice. A portion o...,True,True,Nothing that is a soda is a juice,A portion of the things that are beverages ar...,The only logical conclusion is that some beve...,E soda juice,A beverage thing,A soda beverage
2,e30b1f83-a4c3-49cb-8aaf-5f64208c625b,Everything that is a planet is a celestial bod...,False,False,Everything that is a planet is a celestial body,Anything that is a sun is a celestial body,There exists at least one sun that is a planet,A planet celestial_body,A sun celestial_body,A sun planet
3,a30e07d5-0fb3-4097-9892-4b145b0c54f5,Every cat is an invisible creature. A number o...,True,False,Every cat is an invisible creature,A number of cats are animals,"Consequently, a portion of animals are invisible",A cat creature,E cat animal,E animal animal
4,5b8161b7-b1bf-4e16-a854-cd52cdce8a1b,There are no capital cities which are oceans. ...,True,True,There are no capital cities which are oceans,A select few large cities are classified as c...,"Therefore, it is clear that some large cities...",E capital-city ocean,A city capital,A city ocean
...,...,...,...,...,...,...,...,...,...,...
955,6df0ab4c-8518-4efb-98d5-e1bd172b5a35,Some creatures that can fly are birds. It is t...,False,True,Some creatures that can fly are birds,It is true that some sparrows are flying crea...,"For this reason, every single sparrow is a bird",A bird creature,A sparrow creature,A sparrow bird
956,0db674a5-4391-43fd-8d40-b6ec2b6e77e3,Every single thing that is a hammer is a tool....,False,True,Every single thing that is a hammer is a tool,All screwdrivers are also tools,It must be true that no screwdriver is a hammer,A hammer tool,A screwdriver tool,E hammer screwdriver
957,6650eef5-2045-4ee2-8fe1-a3036d54c6b6,No person who is a doctor is also a teacher. A...,True,True,No person who is a doctor is also a teacher,Anything that is a teacher is a profession,"Therefore, some professions are not doctors",E doctor person,A teacher profession,A doctor profession
958,452c2309-abd1-4cd9-9cd9-1e693ef85cf6,A creature that is a reptile is never a mammal...,True,False,A creature that is a reptile is never a mammal,There is a group of reptiles that are also el...,It must logically follow that some elephants ...,E creature reptile,I reptile elephant,I elephant mammal


In [11]:
VALID_SYLLOGISMS = {
    1: {"AAA", "EAE", "AII", "EIO"},
    2: {"EAE", "AEE", "EIO", "AOO"},
    3: {"AAI", "IAI", "AII", "EAO", "OAO", "EIO"},
    4: {"AAI", "AEE", "IAI", "EAO", "EIO"}
}

VALID_MOODS = set().union(*VALID_SYLLOGISMS.values())

df_inp["pred_vald"] = (
    df_inp["can_1"].str[0] +
    df_inp["can_2"].str[0] +
    df_inp["can_conc"].str[0]
).isin(VALID_MOODS).map({True: "true", False: "false"})



In [12]:
df_inp

,id,syllogism,validity,plausibility,premise1,premise2,conclusion,can_1,can_2,can_conc,pred_vald
0,50146f21-d265-4e3a-8d93-8165cdbe89a3,All cars are a type of vehicle. No animal is a...,False,True,All cars are a type of vehicle,No animal is a car,"Therefore, no animal can be a vehicle",A car vehicle,E animal animal,E animal animal,true
1,dfafb4f6-4e1d-4cd5-aeb4-75d36aafdf1a,Nothing that is a soda is a juice. A portion o...,True,True,Nothing that is a soda is a juice,A portion of the things that are beverages ar...,The only logical conclusion is that some beve...,E soda juice,A beverage thing,A soda beverage,false
2,e30b1f83-a4c3-49cb-8aaf-5f64208c625b,Everything that is a planet is a celestial bod...,False,False,Everything that is a planet is a celestial body,Anything that is a sun is a celestial body,There exists at least one sun that is a planet,A planet celestial_body,A sun celestial_body,A sun planet,true
3,a30e07d5-0fb3-4097-9892-4b145b0c54f5,Every cat is an invisible creature. A number o...,True,False,Every cat is an invisible creature,A number of cats are animals,"Consequently, a portion of animals are invisible",A cat creature,E cat animal,E animal animal,true
4,5b8161b7-b1bf-4e16-a854-cd52cdce8a1b,There are no capital cities which are oceans. ...,True,True,There are no capital cities which are oceans,A select few large cities are classified as c...,"Therefore, it is clear that some large cities...",E capital-city ocean,A city capital,A city ocean,false
...,...,...,...,...,...,...,...,...,...,...,...
955,6df0ab4c-8518-4efb-98d5-e1bd172b5a35,Some creatures that can fly are birds. It is t...,False,True,Some creatures that can fly are birds,It is true that some sparrows are flying crea...,"For this reason, every single sparrow is a bird",A bird creature,A sparrow creature,A sparrow bird,true
956,0db674a5-4391-43fd-8d40-b6ec2b6e77e3,Every single thing that is a hammer is a tool....,False,True,Every single thing that is a hammer is a tool,All screwdrivers are also tools,It must be true that no screwdriver is a hammer,A hammer tool,A screwdriver tool,E hammer screwdriver,false
957,6650eef5-2045-4ee2-8fe1-a3036d54c6b6,No person who is a doctor is also a teacher. A...,True,True,No person who is a doctor is also a teacher,Anything that is a teacher is a profession,"Therefore, some professions are not doctors",E doctor person,A teacher profession,A doctor profession,false
958,452c2309-abd1-4cd9-9cd9-1e693ef85cf6,A creature that is a reptile is never a mammal...,True,False,A creature that is a reptile is never a mammal,There is a group of reptiles that are also el...,It must logically follow that some elephants ...,E creature reptile,I reptile elephant,I elephant mammal,false


In [13]:
df_inp.drop("pred_vald", axis=1)
df_inp

,id,syllogism,validity,plausibility,premise1,premise2,conclusion,can_1,can_2,can_conc,pred_vald
0,50146f21-d265-4e3a-8d93-8165cdbe89a3,All cars are a type of vehicle. No animal is a...,False,True,All cars are a type of vehicle,No animal is a car,"Therefore, no animal can be a vehicle",A car vehicle,E animal animal,E animal animal,true
1,dfafb4f6-4e1d-4cd5-aeb4-75d36aafdf1a,Nothing that is a soda is a juice. A portion o...,True,True,Nothing that is a soda is a juice,A portion of the things that are beverages ar...,The only logical conclusion is that some beve...,E soda juice,A beverage thing,A soda beverage,false
2,e30b1f83-a4c3-49cb-8aaf-5f64208c625b,Everything that is a planet is a celestial bod...,False,False,Everything that is a planet is a celestial body,Anything that is a sun is a celestial body,There exists at least one sun that is a planet,A planet celestial_body,A sun celestial_body,A sun planet,true
3,a30e07d5-0fb3-4097-9892-4b145b0c54f5,Every cat is an invisible creature. A number o...,True,False,Every cat is an invisible creature,A number of cats are animals,"Consequently, a portion of animals are invisible",A cat creature,E cat animal,E animal animal,true
4,5b8161b7-b1bf-4e16-a854-cd52cdce8a1b,There are no capital cities which are oceans. ...,True,True,There are no capital cities which are oceans,A select few large cities are classified as c...,"Therefore, it is clear that some large cities...",E capital-city ocean,A city capital,A city ocean,false
...,...,...,...,...,...,...,...,...,...,...,...
955,6df0ab4c-8518-4efb-98d5-e1bd172b5a35,Some creatures that can fly are birds. It is t...,False,True,Some creatures that can fly are birds,It is true that some sparrows are flying crea...,"For this reason, every single sparrow is a bird",A bird creature,A sparrow creature,A sparrow bird,true
956,0db674a5-4391-43fd-8d40-b6ec2b6e77e3,Every single thing that is a hammer is a tool....,False,True,Every single thing that is a hammer is a tool,All screwdrivers are also tools,It must be true that no screwdriver is a hammer,A hammer tool,A screwdriver tool,E hammer screwdriver,false
957,6650eef5-2045-4ee2-8fe1-a3036d54c6b6,No person who is a doctor is also a teacher. A...,True,True,No person who is a doctor is also a teacher,Anything that is a teacher is a profession,"Therefore, some professions are not doctors",E doctor person,A teacher profession,A doctor profession,false
958,452c2309-abd1-4cd9-9cd9-1e693ef85cf6,A creature that is a reptile is never a mammal...,True,False,A creature that is a reptile is never a mammal,There is a group of reptiles that are also el...,It must logically follow that some elephants ...,E creature reptile,I reptile elephant,I elephant mammal,false


In [16]:
VALID_SYLLOGISMS = {
    1: {"AAA", "EAE", "AII", "EIO"},
    2: {"EAE", "AEE", "EIO", "AOO"},
    3: {"AAI", "IAI", "AII", "EAO", "OAO", "EIO"},
    4: {"AAI", "AEE", "IAI", "EAO", "EIO"}
}

def parse_can(x):
    if not isinstance(x, str):
        return None
    parts = x.strip().split()
    if len(parts) != 3:
        return None
    return parts[0], parts[1], parts[2]


def predict_validity(can1, can2, can_conc):
    p1 = parse_can(can1)
    p2 = parse_can(can2)
    pc = parse_can(can_conc)

    # If any premise/conclusion is malformed → invalid
    if p1 is None or p2 is None or pc is None:
        return False

    q1, s1, p1_ = p1
    q2, s2, p2_ = p2
    qc, sc, pc_ = pc

    S = sc
    P = pc_

    # Middle term
    terms = {s1, p1_, s2, p2_}
    middle = list(terms - {S, P})
    if len(middle) != 1:
        return False
    M = middle[0]

    # Determine figure
    if (s1 == M and p1_ == P) and (s2 == S and p2_ == M):
        figure = 1
    elif (s1 == P and p1_ == M) and (s2 == S and p2_ == M):
        figure = 2
    elif (s1 == M and p1_ == P) and (s2 == M and p2_ == S):
        figure = 3
    elif (s1 == P and p1_ == M) and (s2 == M and p2_ == S):
        figure = 4
    else:
        return False

    mood = q1 + q2 + qc
    return mood in VALID_SYLLOGISMS[figure]


df_inp["pred_vald"] = df_inp.apply(
    lambda r: predict_validity(r["can_1"], r["can_2"], r["can_conc"]),
    axis=1
)

df_inp

,id,syllogism,validity,plausibility,premise1,premise2,conclusion,can_1,can_2,can_conc,pred_vald
0,50146f21-d265-4e3a-8d93-8165cdbe89a3,All cars are a type of vehicle. No animal is a...,False,True,All cars are a type of vehicle,No animal is a car,"Therefore, no animal can be a vehicle",A car vehicle,E animal animal,E animal animal,False
1,dfafb4f6-4e1d-4cd5-aeb4-75d36aafdf1a,Nothing that is a soda is a juice. A portion o...,True,True,Nothing that is a soda is a juice,A portion of the things that are beverages ar...,The only logical conclusion is that some beve...,E soda juice,A beverage thing,A soda beverage,False
2,e30b1f83-a4c3-49cb-8aaf-5f64208c625b,Everything that is a planet is a celestial bod...,False,False,Everything that is a planet is a celestial body,Anything that is a sun is a celestial body,There exists at least one sun that is a planet,A planet celestial_body,A sun celestial_body,A sun planet,False
3,a30e07d5-0fb3-4097-9892-4b145b0c54f5,Every cat is an invisible creature. A number o...,True,False,Every cat is an invisible creature,A number of cats are animals,"Consequently, a portion of animals are invisible",A cat creature,E cat animal,E animal animal,False
4,5b8161b7-b1bf-4e16-a854-cd52cdce8a1b,There are no capital cities which are oceans. ...,True,True,There are no capital cities which are oceans,A select few large cities are classified as c...,"Therefore, it is clear that some large cities...",E capital-city ocean,A city capital,A city ocean,False
...,...,...,...,...,...,...,...,...,...,...,...
955,6df0ab4c-8518-4efb-98d5-e1bd172b5a35,Some creatures that can fly are birds. It is t...,False,True,Some creatures that can fly are birds,It is true that some sparrows are flying crea...,"For this reason, every single sparrow is a bird",A bird creature,A sparrow creature,A sparrow bird,False
956,0db674a5-4391-43fd-8d40-b6ec2b6e77e3,Every single thing that is a hammer is a tool....,False,True,Every single thing that is a hammer is a tool,All screwdrivers are also tools,It must be true that no screwdriver is a hammer,A hammer tool,A screwdriver tool,E hammer screwdriver,False
957,6650eef5-2045-4ee2-8fe1-a3036d54c6b6,No person who is a doctor is also a teacher. A...,True,True,No person who is a doctor is also a teacher,Anything that is a teacher is a profession,"Therefore, some professions are not doctors",E doctor person,A teacher profession,A doctor profession,False
958,452c2309-abd1-4cd9-9cd9-1e693ef85cf6,A creature that is a reptile is never a mammal...,True,False,A creature that is a reptile is never a mammal,There is a group of reptiles that are also el...,It must logically follow that some elephants ...,E creature reptile,I reptile elephant,I elephant mammal,False
